In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

In [5]:
df = pd.read_csv('C:\\Users\\Liman\\Downloads\\Electric_cars_dataset.csv')
df.head()

,ID,VIN (1-10),County,City,State,ZIP Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Base MSRP,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,Expected Price ($1k)
0,EV33174,5YJ3E1EC6L,Snohomish,LYNNWOOD,WA,98037.0,2020.0,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,308,0,32.0,109821694,POINT (-122.287614 47.83874),PUGET SOUND ENERGY INC,50
1,EV40247,JN1AZ0CP8B,Skagit,BELLINGHAM,WA,98229.0,2011.0,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73,0,40.0,137375528,POINT (-122.414936 48.709388),PUGET SOUND ENERGY INC,15
2,EV12248,WBY1Z2C56F,Pierce,TACOMA,WA,98422.0,2015.0,BMW,I3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,81,0,27.0,150627382,POINT (-122.396286 47.293138),BONNEVILLE POWER ADMINISTRATION||CITY OF TACOM...,18
3,EV55713,1G1RD6E44D,King,REDMOND,WA,98053.0,2013.0,CHEVROLET,VOLT,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,38,0,45.0,258766301,POINT (-122.024951 47.670286),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),33.9
4,EV28799,1G1FY6S05K,Pierce,PUYALLUP,WA,98375.0,2019.0,CHEVROLET,BOLT EV,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,238,0,25.0,296998138,POINT (-122.321062 47.103797),BONNEVILLE POWER ADMINISTRATION||CITY OF TACOM...,41.78


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64353 entries, 0 to 64352
Data columns (total 18 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   ID                                                 64353 non-null  object 
 1   VIN (1-10)                                         64353 non-null  object 
 2   County                                             64349 non-null  object 
 3   City                                               64344 non-null  object 
 4   State                                              64342 non-null  object 
 5   ZIP Code                                           64347 non-null  float64
 6   Model Year                                         64346 non-null  float64
 7   Make                                               64349 non-null  object 
 8   Model                                              64340 non-null  object 
 9   Electr

In [7]:
df.isnull().sum()

ID                                                     0
VIN (1-10)                                             0
County                                                 4
City                                                   9
State                                                 11
ZIP Code                                               6
Model Year                                             7
Make                                                   4
Model                                                 13
Electric Vehicle Type                                  0
Clean Alternative Fuel Vehicle (CAFV) Eligibility      0
Electric Range                                         0
Base MSRP                                              0
Legislative District                                 169
DOL Vehicle ID                                         0
Vehicle Location                                     510
Electric Utility                                     722
Expected Price ($1k)           

In [8]:
for col in df.columns :
    if df[col].dtype in ['float64', 'int64'] :
        df[col] = df[col].fillna(df[col].mean())
    else :
        df[col] = df[col].fillna(df[col].mode()[0])    
        

In [9]:
df.isnull().sum()

ID                                                   0
VIN (1-10)                                           0
County                                               0
City                                                 0
State                                                0
ZIP Code                                             0
Model Year                                           0
Make                                                 0
Model                                                0
Electric Vehicle Type                                0
Clean Alternative Fuel Vehicle (CAFV) Eligibility    0
Electric Range                                       0
Base MSRP                                            0
Legislative District                                 0
DOL Vehicle ID                                       0
Vehicle Location                                     0
Electric Utility                                     0
Expected Price ($1k)                                 0
dtype: int

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
num_cols = [
    "Model Year",
    "Electric Range",
    "Base MSRP",
    "Expected Price ($1k)"
]

In [12]:
df = df[(df["Model Year"] >= 2005) & (df["Model Year"] <= 2025)]


In [13]:
Q1 = df["Base MSRP"].quantile(0.25)
Q3 = df["Base MSRP"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df["Base MSRP"] = df["Base MSRP"].clip(lower, upper)


In [14]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
df[["Model Year", "Electric Range", "Base MSRP"]] = scaler.fit_transform(
    df[["Model Year", "Electric Range", "Base MSRP"]]
)


In [15]:
cols_to_drop = ['ID', 'VIN (1-10)', 'DOL Vehicle ID', 'Vehicle Location', 'Legislative District']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

le = LabelEncoder()

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded {col}")

display(df.head())

Encoded County
Encoded City
Encoded State
Encoded Make
Encoded Model
Encoded Electric Vehicle Type
Encoded Clean Alternative Fuel Vehicle (CAFV) Eligibility
Encoded Electric Utility
Encoded Expected Price ($1k)


,County,City,State,ZIP Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Base MSRP,Electric Utility,Expected Price ($1k)
0,120,271,35,98037.0,0.50,27,56,0,0,1.169154,0.0,65,150
1,118,38,35,98229.0,-1.75,21,52,0,0,0.000000,0.0,65,26
2,95,473,35,98422.0,-0.75,3,44,0,0,0.039801,0.0,20,39
3,60,391,35,98053.0,-1.25,5,94,1,0,-0.174129,0.0,66,111
4,95,380,35,98375.0,0.25,5,15,0,0,0.820896,0.0,16,132


In [16]:
X = df.drop(columns=['Expected Price ($1k)'])
y = df['Expected Price ($1k)']

In [17]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)


In [18]:
# svm_model = SVR(kernel='rbf', C=99, gamma=0.22, epsilon=0.1)

In [19]:
# svm_model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = svm_model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)